In [35]:
# Import libraries
import pandas as pd
import numpy as np
import geopandas as gp
import sqlalchemy as sql
import json

import matplotlib.pyplot as plt
import seaborn as sns
import pymssql


In [36]:
# Clear this before publishing
engine = sql.create_engine('mssql+pymssql://DDAMWSQL16/demographic_warehouse')
engine2 = sql.create_engine('mssql+pymssql://sql2014b8/GeoDepot')

In [37]:
# settings
pd.set_option('display.max_columns', None)
%matplotlib inline

pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [38]:
xls = pd.ExcelFile('Forecasted Employment, Regional Controls for 2022 through 2060_Sept 2023 Revised (2).xlsx') #April 9
df1 = pd.read_excel(xls,  '1b. Summary Forecast_Production' )
# df2 = pd.read_excel(xls, 'Sheet2' )
# df3 = pd.read_excel(xls, 'Sheet3' )
# df1['jobs'] = df1['Primary Jobs (count)']#*1000
df1.head()

,Industry,2022,2026,2029,2032,2035,2040,2050,2060
0,"Forestry, fishing, and hunting",9.119,8.947,9.079,9.368,9.603,9.733,9.615,9.441
1,Mining,1.250,1.290,1.309,1.325,1.325,1.315,1.266,1.223
2,Utilities,5.186,4.980,4.813,4.690,4.556,4.280,3.645,3.050
3,Construction,71.263,70.396,72.894,75.706,76.357,77.051,73.604,69.884
4,Manufacturing,135.163,130.086,131.876,121.088,128.868,141.323,157.522,171.409


In [39]:
# cols = ['Category', 'Race', 'Units', 2022]
# lbf = df2[cols]
# lbf['lbfpop'] = lbf[2022]*1000
# lbf

In [40]:
#df3.head()

In [41]:
# Race for popsim
def get_indxwalk(row):
        if row['Industry'] in ['State and Local Government', 'Federal Civilian']:
            return 'job_01'
        elif row['Industry'] in ['Federal Military']:
            return 'job_02'
        elif row['Industry'] in ['Forestry, fishing, and hunting', 'Farm', 'Mining']:
            return 'job_03'
        elif row['Industry'] in ['Information',
                                'Professional, scientific, and technical services', 
                                 'Administrative, support, waste management, and remediation services']:
            return 'job_04'
        elif row['Industry'] in ['Finance and insurance','Real estate and rental and leasing',
                                 'Management of companies and enterprises']:
            return 'job_05'        
        elif row['Industry'] in ['Educational services; private']:
            return 'job_06'
        elif row['Industry'] in ['Health care and social assistance']:
            return 'job_07'
        elif row['Industry'] in ['Retail trade']:
            return 'job_08'
        elif row['Industry'] in ['Construction','Transportation and warehousing' ]:
            return 'job_09'
        elif row['Industry'] in ['Utilities','Manufacturing', 'Wholesale trade']:
            return 'job_10'
        elif row['Industry'] in ['Arts, entertainment, and recreation']:
            return 'job_11'
        elif row['Industry'] in ['Accommodation']:
            return 'job_12'
        elif row['Industry'] in ['Food Service']:
            return 'job_13'
        elif row['Industry'] in ['Other services (except public administration)']:
            return 'job_14'


df1['job_cat'] = df1.apply(get_indxwalk, axis = 1)
df1


,Industry,2022,2026,2029,2032,2035,2040,2050,2060,job_cat
0,"Forestry, fishing, and hunting",9.119,8.947,9.079,9.368,9.603,9.733,9.615,9.441,job_03
1,Mining,1.250,1.290,1.309,1.325,1.325,1.315,1.266,1.223,job_03
2,Utilities,5.186,4.980,4.813,4.690,4.556,4.280,3.645,3.050,job_10
3,Construction,71.263,70.396,72.894,75.706,76.357,77.051,73.604,69.884,job_09
4,Manufacturing,135.163,130.086,131.876,121.088,128.868,141.323,157.522,171.409,job_10
5,Wholesale trade,36.670,35.463,35.142,35.934,35.673,35.240,33.736,31.374,job_10
6,Retail trade,111.954,102.216,100.137,102.018,101.568,101.074,97.944,92.248,job_08
7,Transportation and warehousing,50.227,54.517,56.039,57.090,57.693,58.549,58.876,58.386,job_09
8,Information,21.267,20.757,20.332,20.278,19.908,19.416,18.430,17.293,job_04
9,Finance and insurance,66.658,66.801,66.226,66.540,66.030,64.527,59.964,54.903,job_05


In [42]:
temp = df1.groupby(['job_cat']).agg({2035: 'sum'}).reset_index()
temp['jobs'] = temp[2035]*1000

In [43]:
temp['jobs'].sum()

1682423.97320027

In [44]:
# reading the gQ pop from concep
query_h = '''
  SELECT sum(gq_mil) FROM [sr15_dev].[capacity_outputs].[mgrabase]
WHERE increment = 2035
'''
gqmil_df = pd.read_sql(query_h, con=engine.connect())
print("gqmil_df shape: ", gqmil_df.shape)
gqmil_jobs = gqmil_df.iloc[0, 0]
print("gqmil_jobs: ", gqmil_jobs)

gqmil_df shape:  (1, 1)
gqmil_jobs:  42074


In [45]:
temp = temp.sort_values('job_cat', ascending = True)
temp_t = temp.set_index('job_cat').transpose()
temp_t.loc['jobs', 'job_02'] = temp_t.loc['jobs', 'job_02'] - gqmil_jobs
temp_t['region'] = 1
temp_t.set_index('region', inplace = True)
temp_t.columns.name = None
temp_t

,job_01,job_02,job_03,job_04,job_05,job_06,job_07,job_08,job_09,job_10,job_11,job_12,job_13,job_14
region,,,,,,,,,,,,,,
1,102.394,104.000,10.929,303.373,155.972,135.711,190.045,101.568,134.051,169.097,46.767,16.735,134.234,77.549
1,102394.424,61926.000,10928.511,303372.834,155971.771,135711.418,190044.570,101567.954,134050.589,169096.646,46767.268,16734.767,134234.390,77548.832


In [46]:
temp_t.to_csv('regional_control_ind_2035.csv')

In [47]:
1494959.2307079523 - 41841

1453118.2307079523